# TensorFlow 2.x Architecture: Eager Execution, tf.function, AutoGraph, and Graphs

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/tensorflow_1)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q tensorflow mlflow

In [ ]:
import tensorflow as tf

print(tf.__version__)

# Operations execute immediately
a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf.constant([[5.0, 6.0], [7.0, 8.0]])
c = tf.matmul(a, b)

print(type(c))
print(c)

In [ ]:
import tensorflow as tf

def relu_manual(x):
    # This is real Python control flow — runs eagerly
    if float(x) > 0:
        return x
    return tf.zeros_like(x)

print(relu_manual(tf.constant(3.0)))
print(relu_manual(tf.constant(-1.0)))

In [ ]:
import tensorflow as tf
import time

@tf.function
def matrix_mult_graph(a, b):
    return tf.matmul(a, b)

def matrix_mult_eager(a, b):
    return tf.matmul(a, b)

a = tf.random.normal([1000, 1000])
b = tf.random.normal([1000, 1000])

# Warm up
matrix_mult_graph(a, b)

N = 100

start = time.time()
for _ in range(N):
    matrix_mult_eager(a, b)
eager_time = time.time() - start

start = time.time()
for _ in range(N):
    matrix_mult_graph(a, b)
graph_time = time.time() - start

print(f"Eager: {eager_time:.3f}s")
print(f"Graph: {graph_time:.3f}s")
print(f"Speedup: {eager_time / graph_time:.1f}x")

In [ ]:
import tensorflow as tf

@tf.function
def square_add(x, y):
    return x ** 2 + y

# First call: traces the function, then executes
result = square_add(tf.constant(3.0), tf.constant(1.0))
print(result)

# Subsequent calls with same input signature: uses cached graph
result2 = square_add(tf.constant(4.0), tf.constant(2.0))
print(result2)

In [ ]:
import tensorflow as tf

@tf.function
def add_and_relu(x):
    return tf.nn.relu(x + 1.0)

# Get the concrete function for a specific signature
cf = add_and_relu.get_concrete_function(tf.TensorSpec(shape=[None], dtype=tf.float32))

# Print the graph operations
for op in cf.graph.get_operations():
    print(op.name, op.type)

In [ ]:
import tensorflow as tf

@tf.function
def abs_value(x):
    # This Python if gets converted to tf.cond by AutoGraph
    if x > 0:
        return x
    else:
        return -x

result_pos = abs_value(tf.constant(3.0))
result_neg = abs_value(tf.constant(-5.0))

print(result_pos, result_neg)

In [ ]:
import tensorflow as tf

@tf.function
def bad_branch(x, use_relu: bool):
    # use_relu is a Python bool — evaluated at TRACE time, not runtime
    if use_relu:
        return tf.nn.relu(x)
    else:
        return tf.nn.sigmoid(x)

# Traced with use_relu=True → graph only contains relu
result_relu = bad_branch(tf.constant([-1.0, 1.0]), True)

# Traced with use_relu=False → NEW trace, new graph
result_sigmoid = bad_branch(tf.constant([-1.0, 1.0]), False)

print(result_relu)
print(result_sigmoid)

In [ ]:
import tensorflow as tf

@tf.function
def branching(x, flag: bool):
    return tf.nn.relu(x) if flag else x

# Force retracing by varying the Python bool
for val in [True, False, True, False]:
    branching(tf.constant(1.0), val)

print(f"Number of concrete functions: {len(branching._list_all_concrete_functions())}")

In [ ]:
import tensorflow as tf

@tf.function
def scale(x, factor):
    return x * factor

# Each unique Python int triggers a retrace!
for i in range(5):
    scale(tf.constant(1.0), i)  # retraces 5 times

print(f"Concrete functions: {len(scale._list_all_concrete_functions())}")

In [ ]:
import tensorflow as tf

@tf.function
def scale_fixed(x, factor):
    return x * factor

factor_tensor = tf.constant(3.0)
result = scale_fixed(tf.constant(2.0), factor_tensor)
print(result)

In [ ]:
import tensorflow as tf

@tf.function(input_signature=[
    tf.TensorSpec(shape=[None, 10], dtype=tf.float32),
    tf.TensorSpec(shape=[10, 5], dtype=tf.float32),
])
def linear(x, w):
    return tf.matmul(x, w)

# Works: batch size can vary (None dimension)
result_small = linear(tf.ones([4, 10]), tf.ones([10, 5]))
result_large = linear(tf.ones([100, 10]), tf.ones([10, 5]))

print(result_small.shape, result_large.shape)

# Fails: wrong dtype
try:
    linear(tf.ones([4, 10], dtype=tf.float64), tf.ones([10, 5], dtype=tf.float64))
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10),
])
optimizer = tf.keras.optimizers.Adam(0.001)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

@tf.function  # Only this inner step is graphified
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        logits = model(x_batch, training=True)
        loss = loss_fn(y_batch, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

# Outer loop stays eager — easy to add logging, early stopping, etc.
x_dummy = tf.random.normal([32, 20])
y_dummy = tf.random.uniform([32], maxval=10, dtype=tf.int32)

for step in range(3):
    loss = train_step(x_dummy, y_dummy)
    print(f"Step {step}: loss = {loss:.4f}")